# Li2026 (Triple-N) packaging validation -- from the Brain-Score packaged data

Every figure here is regenerated from `load_dataset('Li2026')`,
`load_dataset('Li2026.temporal')`, and `load_stimulus_set('Li2026')` -- i.e. the
assemblies served from S3, not the raw `.mat` files. Reproducing the paper's figures
from these proves two things at once: the packaging round-trip is lossless, and the
assembly carries enough metadata (region, patch label, reliability, time bins,
stimulus links) to support the dataset's own analyses.

Figures: 1f (reliability by region), 2f (cross-area similarity), 3d (response-type
clusters), 3e (population RSA over time), 5 (AlexNet vs MPNet encoding).
Fig 2g (trial-noise covariance) is intentionally omitted -- it needs trial-resolved
data, which this package deliberately does not carry (responses are trial-averaged).

In [1]:
import os, json, warnings
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # single-threaded BLAS: avoid nbconvert/OpenBLAS deadlock
import numpy as np, pandas as pd, xarray as xr
from scipy.stats import pearsonr
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import brainscore_vision
from brainscore_vision import load_dataset, load_stimulus_set

OUT = os.environ.get("VAL_OUT", "/home/ec2-user/val/figs"); os.makedirs(OUT, exist_ok=True)
order = ["V1", "V2", "V4", "IT"]
SUMMARY = {"source": "brainscore packaged data (load_dataset / load_stimulus_set)"}

## Load the packaged static assembly (per-unit responses + metadata)

In [2]:
static = load_dataset("Li2026").squeeze("time_bin")
static = static.transpose("neuroid", "presentation")
resp = static.values                                  # (neuroid, 1000) trial-averaged responses
region = static["region"].values                      # V1/V2/V4/IT (+ EVC-other)
patch = static["arealabel"].values                    # category-region patch label (MBody, MFace, ...)
rel = static["reliability"].values                    # packaged split-half reliability
sids = static["stimulus_id"].values                   # presentation order (== stimulus_set order)
print("packaged static assembly:", dict(static.sizes))
print("coords:", list(static.coords))
print("reliable >0.4:", int((rel > 0.4).sum()))

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_Assembly.nc:   0%|          | 0.00/135M [00:00<?, ?B/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_Assembly.nc:   4%|▍         | 5.24M/135M [00:00<00:02, 52.3MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_Assembly.nc:  40%|████      | 54.5M/135M [00:00<00:00, 311MB/s] 

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_Assembly.nc:  41%|████      | 54.8M/135M [00:00<00:00, 393MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_Assembly.nc:  70%|███████   | 95.2M/135M [00:00<00:00, 388MB/s]

...

brainscore-storage/brainscore-vision/benchmarks/Li2026/stimulus_Li2026_Stimuli.zip:  69%|██████▉   | 225M/328M [00:00<00:00, 482MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/stimulus_Li2026_Stimuli.zip:  87%|████████▋ | 285M/328M [00:00<00:00, 515MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/stimulus_Li2026_Stimuli.zip:  87%|████████▋ | 285M/328M [00:00<00:00, 538MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/stimulus_Li2026_Stimuli.zip: 100%|██████████| 328M/328M [00:00<00:00, 467MB/s]

packaged static assembly: {'neuroid': 47503, 'presentation': 1000}
coords: ['presentation', 'neuroid', 'time_bin']
reliable >0.4: 35494


## Fig 1f -- split-half reliability by region

In [3]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.boxplot([rel[(region == r) & np.isfinite(rel)] for r in order], tick_labels=order, showfliers=False)
ax.axhline(0.4, color="r", ls="--", lw=1)
ax.set_ylabel("split-half reliability"); ax.set_title("Fig 1f (packaged): reliability by region")
fig.tight_layout(); fig.savefig(f"{OUT}/fig1f_reliability.png", dpi=120); plt.close(fig)
SUMMARY["1f_reliability_median"] = {r: round(float(np.nanmedian(rel[region == r])), 3) for r in order}
SUMMARY["1f_n_reliable"] = {r: int((rel[region == r] > 0.4).sum()) for r in order}
print("Fig1f median reliability:", SUMMARY["1f_reliability_median"])
print("Fig1f reliable counts (paper IT=26700):", SUMMARY["1f_n_reliable"])

Fig1f median reliability: {'V1': 0.7, 'V2': 0.802, 'V4': 0.796, 'IT': 0.604}
Fig1f reliable counts (paper IT=26700): {'V1': 2556, 'V2': 2625, 'V4': 3559, 'IT': 26700}


## Fig 2f -- cross-area similarity (Pearson r between mean response profiles)

In [4]:
rmask = rel > 0.4
def cat(p):
    for k, v in [("MB", 0), ("AB", 0), ("MF", 1), ("AF", 1), ("PF", 1), ("MO", 2), ("AO", 2),
                 ("CLC", 3), ("AMC", 3), ("LPP", 4), ("PITP", 4)]:
        if str(p).startswith(k): return v
    return 5
prof = {p: np.nanmean(resp[rmask & (patch == p)], axis=0) for p in pd.unique(patch[rmask])
        if (rmask & (patch == p)).sum() >= 20 and cat(p) < 5}
plist = sorted(prof, key=lambda p: (cat(p), p)); C = np.corrcoef(np.array([prof[p] for p in plist]))
fig, ax = plt.subplots(figsize=(7, 6)); im = ax.imshow(C, cmap="RdBu_r", vmin=-0.6, vmax=0.6)
ax.set_xticks(range(len(plist))); ax.set_xticklabels(plist, rotation=90, fontsize=6)
ax.set_yticks(range(len(plist))); ax.set_yticklabels(plist, fontsize=6)
ax.set_title("Fig 2f (packaged): cross-area similarity"); fig.colorbar(im, label="Pearson r")
fig.tight_layout(); fig.savefig(f"{OUT}/fig2f_crossarea.png", dpi=120); plt.close(fig)
same = [C[i, j] for i in range(len(plist)) for j in range(i+1, len(plist)) if cat(plist[i]) == cat(plist[j])]
cross = [C[i, j] for i in range(len(plist)) for j in range(i+1, len(plist)) if cat(plist[i]) != cat(plist[j])]
SUMMARY["2f_within_vs_cross_r"] = [round(float(np.mean(same)), 3), round(float(np.mean(cross)), 3)]
SUMMARY["2f_n_patches"] = len(plist)
print(f"Fig2f within-category r={np.mean(same):.3f} vs cross-category r={np.mean(cross):.3f} (paper: within>>cross)")

Fig2f within-category r=0.380 vs cross-category r=0.025 (paper: within>>cross)


## Load the packaged temporal assembly (per-image PSTHs) for Fig 3d, 3e

In [5]:
T = load_dataset("Li2026.temporal")
treg = T["region"].values; trel = T["reliability"].values; tstart = T["time_bin_start"].values
print("packaged temporal assembly:", dict(T.sizes), "regions:", np.unique(treg))

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc:   0%|          | 0.00/2.10G [00:00<?, ?B/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc:   0%|          | 3.15M/2.10G [00:00<01:06, 31.3MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc:   0%|          | 6.29M/2.10G [00:00<01:16, 27.3MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc:   0%|          | 6.82M/2.10G [00:00<01:14, 27.9MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc:   0%|          | 7.86M/2.10G [00:00<01:14, 27.9MB/s]

...

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc:  96%|█████████▌| 2.02G/2.10G [00:03<00:00, 545MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc:  99%|█████████▉| 2.08G/2.10G [00:04<00:00, 560MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc:  99%|█████████▉| 2.08G/2.10G [00:04<00:00, 571MB/s]

brainscore-storage/brainscore-vision/benchmarks/Li2026/assy_Li2026_temporal_Assembly.nc: 100%|██████████| 2.10G/2.10G [00:04<00:00, 520MB/s]

packaged temporal assembly: {'presentation': 1000, 'neuroid': 47503, 'time_bin': 30} regions: ['EVC-other' 'IT' 'V1' 'V2' 'V4']


## Fig 3d -- response-type clusters (k-means on per-unit PSTH) by region

In [6]:
from sklearn.cluster import KMeans
psth = T.mean("presentation").transpose("neuroid", "time_bin").values
keep = trel > 0.4
z = (psth[keep] - psth[keep].mean(1, keepdims=True)) / (psth[keep].std(1, keepdims=True) + 1e-9)
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(z)
lab = km.labels_; reg_k = np.where(np.isin(treg[keep], ["IT-other"]), "IT", treg[keep])
corder = np.argsort([tstart[np.argmax(km.cluster_centers_[c])] for c in range(3)])
remap = {c: i for i, c in enumerate(corder)}; lab = np.array([remap[l] for l in lab])
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
for c in range(3):
    axs[0].plot(tstart, z[lab == c].mean(0), label=f"cluster {c+1} (n={int((lab==c).sum())})")
axs[0].set_title("Fig 3a (packaged): cluster mean PSTH"); axs[0].set_xlabel("ms"); axs[0].legend(fontsize=7)
bottom = np.zeros(len(order))
for c in range(3):
    frac = [np.mean(lab[reg_k == r] == c) if (reg_k == r).sum() else 0 for r in order]
    axs[1].bar(order, frac, bottom=bottom, label=f"cluster {c+1}"); bottom += frac
axs[1].set_title("Fig 3d (packaged): response-type distribution by region"); axs[1].set_ylabel("fraction"); axs[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3d_clusters.png", dpi=120); plt.close(fig)
SUMMARY["3d_cluster_fraction_by_region"] = {r: [round(float(np.mean(lab[reg_k == r] == c)), 2) for c in range(3)] for r in order}
print("Fig3d cluster fractions by region:", SUMMARY["3d_cluster_fraction_by_region"])

Fig3d cluster fractions by region: {'V1': [0.89, 0.05, 0.05], 'V2': [0.95, 0.04, 0.02], 'V4': [0.82, 0.12, 0.06], 'IT': [0.17, 0.53, 0.3]}


## Fig 3e -- population RSA across time (IT)

In [7]:
itm = (np.isin(treg, ["IT", "IT-other"])) & (trel > 0.4)
Tit = T[{"neuroid": itm}].transpose("time_bin", "presentation", "neuroid").values
nb = Tit.shape[0]; rdms = []
for t in range(nb):
    X = Tit[t]; Xc = X - X.mean(1, keepdims=True)
    cc = np.corrcoef(Xc)
    rdms.append(1 - cc[np.triu_indices(cc.shape[0], 1)])
rdms = np.array(rdms); S = np.corrcoef(rdms)
fig, ax = plt.subplots(figsize=(5, 4)); im = ax.imshow(S, origin="lower", cmap="viridis",
    extent=[tstart[0], tstart[-1], tstart[0], tstart[-1]])
ax.contour(S, levels=[0.3, 0.5], colors="w", extent=[tstart[0], tstart[-1], tstart[0], tstart[-1]])
ax.set_title("Fig 3e (packaged): population RSA across time (IT)"); ax.set_xlabel("ms"); ax.set_ylabel("ms")
fig.colorbar(im, label="RDM corr"); fig.tight_layout(); fig.savefig(f"{OUT}/fig3e_rsa_time.png", dpi=120); plt.close(fig)
SUMMARY["3e_rsa_diag_offdiag"] = [round(float(np.mean(np.diag(S, 1))), 3), round(float(S[5, nb-5]), 3)]
print(f"Fig3e RSA near-diagonal r={np.mean(np.diag(S,1)):.3f}, early-vs-late r={S[5,nb-5]:.3f}")

Fig3e RSA near-diagonal r=0.889, early-vs-late r=0.258


## Fig 5 -- AlexNet (FC6) + MPNet encoding from the packaged stimulus set

In [8]:
import torch, torchvision
torch.set_num_threads(1)
from torchvision import transforms
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold

ss = load_stimulus_set("Li2026")
paths = ss.stimulus_paths                                  # stimulus_id -> local image path
sid2coco = dict(zip(ss["stimulus_id"].values, ss["coco_id"].values))
tf = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
net = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.IMAGENET1K_V1).eval()
feat = {}
net.classifier[2].register_forward_hook(lambda m, i, o: feat.__setitem__("fc6", i[0].detach()))
imgs_t = [tf(Image.open(paths[sid]).convert("RGB")) for sid in sids]   # presentation order
X_alex = []
with torch.no_grad():
    for i in range(0, len(imgs_t), 100):
        net(torch.stack(imgs_t[i:i+100])); X_alex.append(feat["fc6"].numpy())
X_alex = np.concatenate(X_alex)
print("alexnet FC6:", X_alex.shape)

def load_captions():
    caps = {}
    for sp in ["train2017", "val2017"]:
        p = os.path.join(os.environ.get("VAL_DATA", "/home/ec2-user/val"), f"captions_{sp}.json")
        if os.path.exists(p):
            for a in json.load(open(p))["annotations"]:
                caps.setdefault(a["image_id"], []).append(a["caption"])
    return caps
X_mpnet = None
try:
    from sentence_transformers import SentenceTransformer
    caps = load_captions()
    if caps:
        mp = SentenceTransformer("all-mpnet-base-v2")
        all_caps, spans = [], []                              # batch all captions into one encode call
        for sid in sids:
            cs = caps.get(int(sid2coco[sid]), [""])[:5] or [""]
            spans.append((len(all_caps), len(all_caps) + len(cs))); all_caps.extend(cs)
        emb = mp.encode(all_caps, batch_size=64, show_progress_bar=False)
        X_mpnet = np.array([emb[a:b].mean(0) for a, b in spans])
        print("mpnet:", X_mpnet.shape)
except Exception as e:
    print("MPNet skipped:", e)

def encode(Xfeat, Y, n_pca=100, n_pls=25, seed=0):
    Xp = PCA(n_components=min(n_pca, Xfeat.shape[1]), random_state=seed).fit_transform(
        (Xfeat - Xfeat.mean(0)) / (Xfeat.std(0) + 1e-9))
    pred = np.zeros_like(Y)
    for tr, te in KFold(10, shuffle=True, random_state=seed).split(Xp):
        pls = PLSRegression(n_components=min(n_pls, Xp.shape[1])).fit(Xp[tr], Y[tr])
        pred[te] = pls.predict(Xp[te])
    return np.array([pearsonr(Y[:, u], pred[:, u])[0] if Y[:, u].std() > 0 else np.nan for u in range(Y.shape[1])])

enc = {}
for r in order:
    mm = (region == r) & (rel > 0.4)
    Y = resp[mm].T                                          # (1000, n_units), presentation order
    enc[(r, "alex")] = encode(X_alex, Y)
    if X_mpnet is not None:
        enc[(r, "mpnet")] = encode(X_mpnet, Y)
SUMMARY["5_alexnet_encoding_median_r"] = {r: round(float(np.nanmedian(enc[(r, "alex")])), 3) for r in order}
print("Fig5 AlexNet encoding median r by region:", SUMMARY["5_alexnet_encoding_median_r"])
if X_mpnet is not None:
    SUMMARY["5_mpnet_encoding_median_r"] = {r: round(float(np.nanmedian(enc[(r, "mpnet")])), 3) for r in order}
    a = enc[("IT", "alex")]; mp_ = enc[("IT", "mpnet")]; v = np.isfinite(a) & np.isfinite(mp_)
    x, y = mp_[v], a[v]; sxx = np.var(x); syy = np.var(y); sxy = np.cov(x, y)[0, 1]
    slope = (syy - sxx + np.sqrt((syy - sxx)**2 + 4*sxy**2)) / (2*sxy)
    SUMMARY["5_LVR_IT"] = round(float(1 / slope), 3)
    print("Fig5 MPNet median r:", SUMMARY["5_mpnet_encoding_median_r"])
    print(f"Fig5 LVR (IT) ~= {SUMMARY['5_LVR_IT']} (paper LVR=0.74, <1 => more visual than semantic)")

fig, ax = plt.subplots(figsize=(6, 3.5))
for r in order:
    ax.hist(enc[(r, "alex")][np.isfinite(enc[(r, "alex")])], bins=40, histtype="step", label=f"{r} AlexNet")
ax.set_xlabel("per-unit encoding accuracy (Pearson r)"); ax.set_title("Fig 5 (packaged): AlexNet encoding accuracy")
ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(f"{OUT}/fig5_encoding.png", dpi=120); plt.close(fig)

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /home/ec2-user/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


  0%|          | 0.00/233M [00:00<?, ?B/s]

 20%|██        | 47.0M/233M [00:00<00:00, 493MB/s]

 40%|████      | 94.0M/233M [00:00<00:00, 475MB/s]

 60%|█████▉    | 139M/233M [00:00<00:00, 475MB/s] 

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

mpnet: (1000, 768)


Fig5 AlexNet encoding median r by region: {'V1': 0.28, 'V2': 0.228, 'V4': 0.276, 'IT': 0.318}
Fig5 MPNet median r: {'V1': 0.135, 'V2': 0.093, 'V4': 0.155, 'IT': 0.222}
Fig5 LVR (IT) ~= 0.908 (paper LVR=0.74, <1 => more visual than semantic)


In [9]:
json.dump(SUMMARY, open(f"{OUT}/validation_summary.json", "w"), indent=2)
print("\n=== VALIDATION SUMMARY (packaged data) ==="); print(json.dumps(SUMMARY, indent=2))
print("VALIDATION_DONE")


=== VALIDATION SUMMARY (packaged data) ===
{
  "source": "brainscore packaged data (load_dataset / load_stimulus_set)",
  "1f_reliability_median": {
    "V1": 0.7,
    "V2": 0.802,
    "V4": 0.796,
    "IT": 0.604
  },
  "1f_n_reliable": {
    "V1": 2556,
    "V2": 2625,
    "V4": 3559,
    "IT": 26700
  },
  "2f_within_vs_cross_r": [
    0.38,
    0.025
  ],
  "2f_n_patches": 22,
  "3d_cluster_fraction_by_region": {
    "V1": [
      0.89,
      0.05,
      0.05
    ],
    "V2": [
      0.95,
      0.04,
      0.02
    ],
    "V4": [
      0.82,
      0.12,
      0.06
    ],
    "IT": [
      0.17,
      0.53,
      0.3
    ]
  },
  "3e_rsa_diag_offdiag": [
    0.889,
    0.258
  ],
  "5_alexnet_encoding_median_r": {
    "V1": 0.28,
    "V2": 0.228,
    "V4": 0.276,
    "IT": 0.318
  },
  "5_mpnet_encoding_median_r": {
    "V1": 0.135,
    "V2": 0.093,
    "V4": 0.155,
    "IT": 0.222
  },
  "5_LVR_IT": 0.908
}
VALIDATION_DONE
